# Spectacular Analysis — Full Tour Notebook

This notebook demonstrates all **17 phases** of the spectacular analysis pipeline
using the `doc_A` golden fixture (opaque ID). It serves as both a **self-study guide** and a
**live demo script** for soutenance presentations.

## Prerequisites

- Conda environment `projet-is` or `projet-is-roo-new` activated
- No API keys required (uses pre-computed golden fixture)
- Data: `tests/golden/fixtures/spectacular/doc_a_golden.json`

In [ ]:
import json
import pathlib
from IPython.display import HTML, display

# Locate golden fixture
FIXTURE_PATH = pathlib.Path("..") / "tests" / "golden" / "fixtures" / "spectacular" / "doc_a_golden.json"
if not FIXTURE_PATH.exists():
    FIXTURE_PATH = pathlib.Path("tests/golden/fixtures/spectacular/doc_a_golden.json")

print(f"Fixture path: {FIXTURE_PATH.resolve()}")
print(f"Exists: {FIXTURE_PATH.exists()}")

In [ ]:
# Load the spectacular analysis state
with open(FIXTURE_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

state = data["state_snapshot"]
capabilities = data.get("capabilities_used", [])

print(f"Workflow: {data['workflow_name']}")
print(f"Source: {data['source_id']} (opaque ID)")
print(f"Phases completed: {data['summary']['completed']}/{data['summary']['total']}")
print(f"Capabilities ({len(capabilities)}): {', '.join(capabilities)}")

---
## Section 1: Argument Extraction & Quality Scoring

Phases: `fact_extraction`, `argument_quality`

The pipeline extracts structured arguments (premises + conclusions) from the source text,
then scores each on 4 quality dimensions: clarity, coherence, relevance, completeness.

In [ ]:
# Display extracted arguments with quality scores
arguments = state["identified_arguments"]
quality = state["argument_quality_scores"]

rows = []
for arg_id, arg_text in arguments.items():
    scores = quality.get(arg_id, {})
    overall = scores.get("overall", 0)
    color = "#27ae60" if overall >= 0.85 else "#f39c12" if overall >= 0.75 else "#e74c3c"
    arg_type = "Premise" if "_p" in arg_id else "Conclusion"
    display_text = arg_text[:80]
    rows.append(
        f"<tr><td><code>{arg_id}</code></td>"
        f"<td>{display_text}</td><td>{arg_type}</td>"
        f"<td>{scores.get('clarity', 0):.2f}</td>"
        f"<td>{scores.get('coherence', 0):.2f}</td>"
        f"<td>{scores.get('relevance', 0):.2f}</td>"
        f"<td>{scores.get('completeness', 0):.2f}</td>"
        f"<td style='color:{color};font-weight:bold'>{overall:.2f}</td></tr>"
    )

n_premises = sum(1 for a in arguments if "_p" in a)
n_conclusions = sum(1 for a in arguments if "_c" in a)
table = (
    "<table style='border-collapse:collapse;width:100%;font-size:0.85em'>"
    "<tr style='background:#2c3e50;color:white'>"
    "<th>ID</th><th>Text</th><th>Type</th>"
    "<th>Clarity</th><th>Coherence</th><th>Relevance</th><th>Completeness</th><th>Overall</th></tr>"
    + "".join(rows) + "</table>"
    f"<p><strong>{len(arguments)}</strong> arguments, "
    f"<strong>{n_premises}</strong> premises, <strong>{n_conclusions}</strong> conclusions</p>"
)
display(HTML(table))

---
## Section 2: Fallacy Detection

Phases: `neural_fallacy_detection`, `hierarchical_fallacy_detection`

Two-tier hybrid detection: neural classification followed by hierarchical taxonomy
(8 families: relevance, causal, distortion, presumption, ambiguity, diversion, erosion, induction).

In [ ]:
# Display detected fallacies with confidence heatmap
fallacies = state["identified_fallacies"]
neural_scores = state.get("neural_fallacy_scores", [])

rows = []
for fid, fdata in fallacies.items():
    conf = fdata["confidence"]
    bg = f"hsl({int((1-conf)*120)}, 70%, 90%)"
    ftype = fdata["type"].replace("_", " ").title()
    rows.append(
        f"<tr style='background:{bg}'>"
        f"<td><code>{fid}</code></td>"
        f"<td><strong>{ftype}</strong></td>"
        f"<td>{fdata['family'].title()}</td>"
        f"<td>{conf:.0%}</td>"
        f"<td><code>{fdata['source_arg']}</code></td>"
        f"<td style='font-size:0.8em'>{fdata['justification']}</td></tr>"
    )

# Taxonomy frequency
family_counts = {}
for fdata in fallacies.values():
    fam = fdata["family"]
    family_counts[fam] = family_counts.get(fam, 0) + 1

freq_bars = "".join(
    f"<div style='margin:2px 0'><span style='display:inline-block;width:{c*60}px;"
    f"background:#e74c3c;color:white;text-align:center;padding:2px 8px;"
    f"border-radius:3px'>{fam.title()}: {c}</span></div>"
    for fam, c in sorted(family_counts.items(), key=lambda x: -x[1])
)

html = (
    f"<h4>Detected Fallacies ({len(fallacies)})</h4>"
    "<table style='border-collapse:collapse;width:100%;font-size:0.85em'>"
    "<tr style='background:#2c3e50;color:white'>"
    "<th>ID</th><th>Type</th><th>Family</th><th>Confidence</th>"
    "<th>Source</th><th>Justification</th></tr>"
    + "".join(rows) + "</table>"
    "<h4>Taxonomy Frequency</h4>" + freq_bars
)
display(HTML(html))

---
## Section 3: Logical Formalization (PL / FOL / Modal)

Phases: `nl_to_logic`, `propositional_logic`, `first_order_logic`, `modal_logic`

Natural language arguments are translated into three formal systems.
Each system is queried for satisfiability, validity, and entailment.

In [ ]:
# Display logic translations and analysis results
translations = state.get("nl_to_logic_translations", [])
propositional = state.get("propositional_analysis_results", [])
fol_results = state.get("fol_analysis_results", [])
modal_results = state.get("modal_analysis_results", [])

# NL-to-Logic translations
trans_rows = "".join(
    f"<tr><td><code>{t['source']}</code></td>"
    f"<td><code>{t['logic']}</code></td><td>{t['method']}</td></tr>"
    for t in translations
)

# Analysis summaries
systems = [
    ("Propositional", propositional),
    ("FOL", fol_results),
    ("Modal (S5)", modal_results),
]
summaries = []
for name, results in systems:
    if results:
        valid = sum(1 for r in results if r.get("valid") is True or r.get("satisfiable") is True)
        summaries.append(f"<li><strong>{name}</strong>: {valid}/{len(results)} satisfiable/valid</li>")

# FOL signature
sig = state.get("fol_signature", [])
sig_display = ", ".join(f"<code>{s}</code>" for s in sig[:12])
if len(sig) > 12:
    sig_display += f", ... (+{len(sig)-12} more)"

html = (
    f"<h4>NL-to-Logic Translations ({len(translations)})</h4>"
    "<table style='border-collapse:collapse;width:100%;font-size:0.85em'>"
    "<tr style='background:#2c3e50;color:white'><th>Source</th><th>Formal</th><th>Method</th></tr>"
    + trans_rows + "</table>"
    "<h4>Analysis Summary</h4><ul>" + "".join(summaries) + "</ul>"
    f"<h4>FOL Signature</h4><p>{sig_display}</p>"
)
display(HTML(html))

---
## Section 4: Dung Argumentation Framework

Phase: `dung_extensions`

Abstract argumentation framework with attack relations.
Extensions computed under grounded, preferred, and stable semantics.

In [ ]:
# Display Dung framework structure
dung = state.get("dung_frameworks", {}).get("fw_complete", {})
attacks = dung.get("attacks", [])
extensions = dung.get("extensions", {})
status_map = dung.get("status_assignment", {})

# Attack relations
attack_rows = "".join(
    f"<tr><td><code>{a['from']}</code></td>"
    f"<td style='color:#e74c3c'>&#x279C;</td>"
    f"<td><code>{a['to']}</code></td></tr>"
    for a in attacks
)

# Extensions
ext_rows = []
for sem, ext in extensions.items():
    if isinstance(ext, list) and ext and isinstance(ext[0], list):
        for i, e in enumerate(ext):
            ext_rows.append(
                f"<tr><td>{sem} #{i+1}</td><td>{len(e)} args</td>"
                f"<td>{', '.join(e[:4])}{'...' if len(e)>4 else ''}</td></tr>"
            )
    elif isinstance(ext, list):
        ext_rows.append(
            f"<tr><td>{sem}</td><td>{len(ext)} args</td>"
            f"<td>{', '.join(ext[:4])}{'...' if len(ext)>4 else ''}</td></tr>"
        )

# Status assignment
status_colors = {
    "skeptically_accepted": "#27ae60",
    "credulously_accepted": "#f39c12",
    "rejected": "#e74c3c",
}
status_str = " ".join(
    f"<span style='background:{status_colors.get(s, '#bbb')};color:white;"
    f"padding:2px 8px;border-radius:3px;font-size:0.8em'>"
    f"{arg}: {s.replace('_', ' ')}</span>"
    for arg, s in status_map.items()
)

html = (
    f"<h4>Attack Relations ({len(attacks)})</h4>"
    "<table style='border-collapse:collapse;font-size:0.85em'>"
    "<tr style='background:#2c3e50;color:white'><th>From</th><th></th><th>To</th></tr>"
    + attack_rows + "</table>"
    "<h4>Extensions</h4>"
    "<table style='border-collapse:collapse;font-size:0.85em'>"
    "<tr style='background:#2c3e50;color:white'><th>Semantics</th><th>Size</th><th>Arguments</th></tr>"
    + "".join(ext_rows) + "</table>"
    f"<h4>Status Assignment</h4><p>{status_str}</p>"
)
display(HTML(html))

---
## Section 5: ATMS Hypothesis Branching

Phase: `assumption_based_reasoning` (ATMS)

The Assumption-based Truth Maintenance System explores multiple reasoning contexts.
Each context combines assumptions to build environments; contradictory contexts reveal nogoods.

In [ ]:
# Display ATMS contexts
contexts = state.get("atms_contexts", [])

ctx_rows = []
for ctx in contexts:
    status_color = "#e74c3c" if ctx["status"] == "contradictory" else "#27ae60"
    nogoods = ", ".join(ctx.get("nogoods", [])) or "\u2014"
    assumptions_str = ", ".join(
        f"<em>{a.replace('assumption_', '')}</em>"
        for a in ctx.get("assumptions", [])
    )
    ctx_rows.append(
        f"<tr><td><strong>{ctx['context_id'].replace('ctx_', '')}</strong>"
        f"<br><span style='font-size:0.8em'>{ctx['label']}</span></td>"
        f"<td>{assumptions_str}</td>"
        f"<td>{len(ctx.get('environment', []))} args</td>"
        f"<td style='color:{status_color};font-weight:bold'>{ctx['status'].upper()}</td>"
        f"<td>{nogoods}</td></tr>"
    )

consistent = sum(1 for c in contexts if c["status"] == "consistent")
contra = sum(1 for c in contexts if c["status"] == "contradictory")

html = (
    f"<h4>ATMS Contexts ({len(contexts)}: {consistent} consistent, {contra} contradictory)</h4>"
    "<table style='border-collapse:collapse;width:100%;font-size:0.85em'>"
    "<tr style='background:#2c3e50;color:white'>"
    "<th>Context</th><th>Assumptions</th><th>Env Size</th><th>Status</th><th>Nogoods</th></tr>"
    + "".join(ctx_rows) + "</table>"
)
display(HTML(html))

---
## Section 6: JTMS Belief Revision & Retraction Cascades

Phase: `jtms_belief_revision`

The Justification-based Truth Maintenance System tracks belief statuses (IN/OUT).
Counter-arguments trigger retraction cascades that propagate through the belief network.

In [ ]:
# Display JTMS beliefs and retraction cascades
beliefs = state.get("jtms_beliefs", {})
cascades = state.get("jtms_retraction_chain", [])
revision = state.get("belief_revision_results", {})

# Belief table
belief_rows = []
for bid, bdata in beliefs.items():
    in_status = bdata["status"]
    color = "#27ae60" if in_status == "IN" else "#e74c3c"
    retracted_by = bdata.get("retracted_by", "")
    belief_rows.append(
        f"<tr><td><code>{bid}</code></td>"
        f"<td style='color:{color};font-weight:bold'>{in_status}</td>"
        f"<td>{bdata['confidence']:.2f}</td>"
        f"<td style='font-size:0.8em'>{bdata['justification']}</td>"
        f"<td>{retracted_by or '\u2014'}</td></tr>"
    )

# Cascade visualization
cascade_blocks = []
for cas in cascades:
    effects = "<br>".join(
        f"{e.get('belief', '')}: {e.get('old_status', e.get('old_confidence', ''))} "
        f"\u2192 {e.get('new_status', e.get('new_confidence', ''))}"
        for e in cas.get("downstream_effects", [])
    )
    retracted_str = ", ".join(cas.get("retracted", [])) or "none (confidence only)"
    cascade_blocks.append(
        f"<div style='border:1px solid #e74c3c;border-radius:5px;padding:8px;"
        f"margin:5px 0;background:#fef5f5'>"
        f"<strong>{cas['cascade_id']}</strong> (trigger: <code>{cas['trigger']}</code>)<br>"
        f"<em>{cas['reason']}</em><br>"
        f"Retracted: {retracted_str}<br>"
        f"Effects: {effects}</div>"
    )

init_beliefs = revision.get("initial_beliefs", "?")
rev_beliefs = revision.get("revised_beliefs", "?")
retractions = revision.get("retractions", 0)
method = revision.get("method", "?")
html = (
    f"<h4>Belief Status ({len(beliefs)} beliefs)</h4>"
    "<table style='border-collapse:collapse;width:100%;font-size:0.85em'>"
    "<tr style='background:#2c3e50;color:white'>"
    "<th>ID</th><th>Status</th><th>Confidence</th><th>Justification</th><th>Retracted By</th></tr>"
    + "".join(belief_rows) + "</table>"
    f"<h4>Retraction Cascades ({len(cascades)})</h4>"
    + "".join(cascade_blocks)
    f"<p>Belief revision: {init_beliefs} \u2192 {rev_beliefs} beliefs, "
    f"{retractions} retractions, method={method}</p>"
)
display(HTML(html))

---
## Section 7: Counter-Argumentation & Governance

Phases: `counter_argumentation`, `debate_protocol`, `governance_simulation`

Four counter-argument strategies are deployed, debated under Walton-Krabbe protocols,
then resolved through multi-method governance voting (majority, Borda, Condorcet).

In [ ]:
# Display counter-arguments, debate, and governance
counters = state.get("counter_arguments", [])
debate = state.get("debate_transcripts", [])
dialogue = state.get("dialogue_results", {})
governance = state.get("governance_decisions", [])

# Counter-arguments
counter_rows = "".join(
    f"<tr><td><strong>{c['strategy'].replace('_', ' ').title()}</strong></td>"
    f"<td><code>{c['target_arg']}</code></td>"
    f"<td style='font-size:0.85em'>{c['counter_content']}</td>"
    f"<td style='font-weight:bold'>{c['strength']:.0%}</td></tr>"
    for c in counters
)

# Debate rounds
debate_rows = "".join(
    f"<tr><td>R{d['round']}</td>"
    f"<td><code>{d['proponent']}</code></td>"
    f"<td style='font-size:0.85em'>{d['proponent_move']}</td>"
    f"<td style='font-size:0.85em'>{d['opponent_move']}</td>"
    f"<td>{'\u2705' if d['winner']=='opponent' else '\u2696' if d['winner']=='split' else '\u274C'}</td></tr>"
    for d in debate
)

# Governance voting
gov_rows = []
for dec in governance:
    consensus = dec.get("consensus_score", 0)
    cc = "#27ae60" if consensus >= 0.75 else "#f39c12"
    gov_rows.append(
        f"<tr><td><strong>{dec['method'].title()}</strong></td>"
        f"<td><code>{dec['result'].replace('_', ' ')}</code></td>"
        f"<td style='color:{cc};font-weight:bold'>{consensus:.0%}</td></tr>"
    )

protocol = dialogue.get("protocol", "?")
total_moves = dialogue.get("total_moves", "?")
winner_str = governance[0]["result"].replace("_", " ") if governance else "N/A"
html = (
    f"<h4>Counter-Arguments ({len(counters)})</h4>"
    "<table style='border-collapse:collapse;width:100%;font-size:0.85em'>"
    "<tr style='background:#2c3e50;color:white'>"
    "<th>Strategy</th><th>Target</th><th>Content</th><th>Strength</th></tr>"
    + counter_rows + "</table>"
    f"<h4>Debate Transcript ({protocol}, {total_moves} moves)</h4>"
    "<table style='border-collapse:collapse;width:100%;font-size:0.85em'>"
    "<tr style='background:#2c3e50;color:white'>"
    "<th>Round</th><th>Proponent</th><th>Move</th><th>Opponent Move</th><th>Winner</th></tr>"
    + debate_rows + "</table>"
    f"<h4>Governance Simulation ({len(governance)} methods)</h4>"
    "<table style='border-collapse:collapse;font-size:0.85em'>"
    "<tr style='background:#2c3e50;color:white'><th>Method</th><th>Winner</th><th>Consensus</th></tr>"
    + "".join(gov_rows) + "</table>"
    f"<p>All {len(governance)} voting methods agree: <strong>{winner_str}</strong></p>"
)
display(HTML(html))

---
## Section 8: Narrative Synthesis

Phase: `narrative_synthesis`

The final phase integrates all 16 preceding phases into a coherent narrative assessment.

In [ ]:
# Display narrative synthesis
synthesis = state.get("narrative_synthesis", "No synthesis available.")
ranking = state.get("ranking_results", [])
formal_synth = state.get("formal_synthesis_reports", [])

# Synthesis text
synth_html = (
    "<div style='background:#f8f9fa;border-left:4px solid #2c3e50;"
    f"padding:12px;margin:10px 0;font-size:0.9em;line-height:1.6'>"
    f"{synthesis}</div>"
)

# Argument ranking
rank_rows = "".join(
    f"<tr><td>#{r['rank']}</td><td><code>{r['argument']}</code></td>"
    f"<td style='font-weight:bold'>{r['score']:.2f}</td></tr>"
    for r in ranking
)

# Formal synthesis
formal_rows = "".join(
    f"<tr><td><strong>{r['logic_system'].title()}</strong></td>"
    f"<td style='font-size:0.85em'>{r['summary']}</td>"
    f"<td style='font-size:0.8em'>{'<br>'.join(r.get('key_findings', []))}</td></tr>"
    for r in formal_synth
)

html = (
    "<h4>Narrative Synthesis</h4>" + synth_html +
    "<h4>Argument Ranking</h4>"
    "<table style='border-collapse:collapse;font-size:0.85em'>"
    "<tr style='background:#2c3e50;color:white'><th>Rank</th><th>Argument</th><th>Score</th></tr>"
    + rank_rows + "</table>"
    "<h4>Formal Synthesis Reports</h4>"
    "<table style='border-collapse:collapse;width:100%;font-size:0.85em'>"
    "<tr style='background:#2c3e50;color:white'><th>System</th><th>Summary</th><th>Key Findings</th></tr>"
    + formal_rows + "</table>"
)
display(HTML(html))

---
## Conclusion

This notebook covered all **17 spectacular analysis phases**:

1. `fact_extraction` — Identified 8 arguments (5 premises, 3 conclusions)
2. `argument_quality` — Scored arguments on 4 dimensions (0.74–0.90)
3. `nl_to_logic` — Translated NL to formal logic (template-based)
4. `neural_fallacy_detection` — Neural fallacy scoring (4 high-confidence)
5. `hierarchical_fallacy_detection` — Taxonomy classification (3 families)
6. `propositional_logic` — PL satisfiability (3/5 valid)
7. `first_order_logic` — FOL entailment (16-predicate signature)
8. `modal_logic` — S5 modal analysis (5 formulas)
9. `dung_extensions` — Abstract argumentation (grounded/preferred/stable)
10. `aspic_analysis` — Structured argumentation (3 rules, strict + defeasible)
11. `counter_argumentation` — 4 counter-arguments (reductio, counter-example, distinction, concession)
12. `jtms_belief_revision` — Belief tracking with 2 retraction cascades
13. `debate_protocol` — Walton-Krabbe PPD (3 rounds, opponent wins)
14. `assumption_based_reasoning` — ATMS (4 contexts, 2 contradictory)
15. `governance_simulation` — Multi-method voting (majority, Borda, Condorcet)
16. `formal_synthesis` — Cross-system synthesis (PL + FOL + Modal)
17. `narrative_synthesis` — Integrated narrative assessment

### Next Steps

- Run the HTML report renderer (B.1) for an interactive standalone report
- Explore other scenario fixtures (B.3)
- Try live analysis with your own text (requires API keys)